<a href="https://colab.research.google.com/github/Malesus/Homicidios-Colombia/blob/main/ACA_3_HOMICIDIOS_COLOMBIA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Desarrollo del Proceso ETL**

## **1. Extracción de Datos (Extract)**

La etapa de extracción consistió en la conexión automatizada entre Google Colab y los archivos CSV almacenados en GitHub. Esta metodología permitió trabajar sobre una fuente centralizada y reutilizable, simulando escenarios reales de almacenamiento empresarial.

Inicialmente, se cargaron las librerías necesarias para el procesamiento de datos:

In [122]:
# --- LIBRERÍAS ---

import pandas as pd
import numpy as np

print("\n" + "—"*35)
print("✅ LIBRERÍAS CARGADAS CON ÉXITO")
print("—"*35)
print(f"   Pandas: v{pd.__version__}")
print(f"   NumPy:  v{np.__version__}")
print("—"*35 + "\n")


———————————————————————————————————
✅ LIBRERÍAS CARGADAS CON ÉXITO
———————————————————————————————————
   Pandas: v2.2.2
   NumPy:  v2.0.2
———————————————————————————————————



Posteriormente, se realizó la conexión con el archivo CSV alojado en GitHub mediante el enlace RAW del repositorio y se verificó la correcta carga de la información:

In [123]:
import requests

URL = "https://raw.githubusercontent.com/Malesus/Homicidios-Colombia/refs/heads/main/Homicidios_Colombia_2017_2023.csv"

try:
    # Prueba la conexión con la URL (espera máximo 5 segundos)
    respuesta = requests.get(URL, timeout=5)

    # Si responde un 200 OK, la URL funciona y lee el dataset
    if respuesta.status_code == 200:
        print("✅ ¡Conexión Exitosa!", "\n")
        # Carga el archivo CSV en la variable df
        df = pd.read_csv(URL)
        print(df.head(), "\n")
    else:
        print(f"❌ Conexión Fallida (Código: {respuesta.status_code})")

except:
    # Entra aquí si no hay internet o si la URL está mal escrita
    print("❌ Conexión Fallida (Error de red o URL inválida)")

✅ ¡Conexión Exitosa! 

  FECHA HECHO  COD_DEPTO DEPARTAMENTO  COD_MUNI   MUNICIPIO    ZONA  \
0  2017-01-01         91     AMAZONAS     91001     LETICIA   RURAL   
1  2017-01-01          5    ANTIOQUIA      5091     BETANIA  URBANA   
2  2017-01-01          5    ANTIOQUIA      5212  COPACABANA  URBANA   
3  2017-01-01          5    ANTIOQUIA      5240     EBEJICO   RURAL   
4  2017-01-01          5    ANTIOQUIA      5250    EL BAGRE  URBANA   

        SEXO                   ARMA MEDIO                 MODALIDAD PRESUNTA  \
0  MASCULINO  ARMA BLANCA / CORTOPUNZANTE                              RIÑAS   
1  MASCULINO                ARMA DE FUEGO                              RIÑAS   
2  MASCULINO  ARMA BLANCA / CORTOPUNZANTE  RIÑA ENTRE COMPAÑEROS PERMANENTES   
3  MASCULINO                ARMA DE FUEGO                          SICARIATO   
4  MASCULINO  ARMA BLANCA / CORTOPUNZANTE                              RIÑAS   

               SPOA_CARACTERIZACION  CANTIDAD  
0  HOMICIDIO INTENCIO

Adicionalmente, se integró una segunda fuente de datos correspondiente a coordenadas geográficas de los departamentos de Colombia.

In [124]:
import requests

URL = "https://raw.githubusercontent.com/Malesus/Homicidios-Colombia/refs/heads/main/C%C3%B3digos_departamentos.csv"

try:
    # Probar conexión
    respuesta = requests.get(URL, timeout=5)

    # Validar conexión exitosa
    if respuesta.status_code == 200:
        print("✅ ¡Conexión Exitosa!", "\n")

        # Cargar nuevo dataset
        df_Departamentos = pd.read_csv(URL)

        # Mostrar primeras filas
        print(df_Departamentos.head(), "\n")

    else:
        print(f"❌ Conexión Fallida (Código: {respuesta.status_code})")

except:
    print("❌ Conexión Fallida (Error de red o URL inválida)")

✅ ¡Conexión Exitosa! 

   Código Departamento Nombre Departamento       longitud       Latitud
0                    5           ANTIOQUIA  -75,504557037   6,702032125
1                    8           ATLÁNTICO  -74,965219492  10,677009534
2                   11        BOGOTÁ, D.C.  -74,181072702   4,316107698
3                   13             BOLÍVAR  -74,235148136   8,079796863
4                   15              BOYACÁ  -72,627880544   5,891672889 



## **2. Transformación de Datos (Transform)**

### **Exploración y Auditoría Inicial de Datos**

Con el objetivo de comprender la estructura del dataset y validar la calidad de la información, se realizó un proceso de exploración inicial de datos.

Durante esta etapa se identificaron:

* Número de registros.
* Cantidad de columnas.
* Tipos de datos.
* Integridad estructural.
* Valores faltantes.
* Distribución de categorías.

### ***Auditoría de Dimensiones***

In [125]:
# Obtiene la cantidad total de filas del DataFrame
total_registros = df.shape[0]

# Obtiene la cantidad total de columnas del DataFrame
total_columnas = df.shape[1]

# Calcula la cantidad total de celdas o elementos de datos
total_elementos = df.size

# Extrae las etiquetas de texto correspondientes a las columnas
nombres_columnas = df.columns

# Imprime en consola los indicadores de estructura y dimensiones calculados
print("\n" + "—"*40)
print("📊 RESUMEN ESTRUCTURAL DEL DATAFRAME")
print("—"*40)
print(f"✅ Filas (Registros):   {total_registros:,}")
print(f"✅ Columnas:           {total_columnas}")
print(f"✅ Tamaño total:       {total_elementos:,} datos")
print("—"*40)
print(f"📋 LISTA DE COLUMNAS:\n{list(nombres_columnas)}")
print("—"*40 + "\n")


————————————————————————————————————————
📊 RESUMEN ESTRUCTURAL DEL DATAFRAME
————————————————————————————————————————
✅ Filas (Registros):   89,548
✅ Columnas:           11
✅ Tamaño total:       985,028 datos
————————————————————————————————————————
📋 LISTA DE COLUMNAS:
['FECHA HECHO', 'COD_DEPTO', 'DEPARTAMENTO', 'COD_MUNI', 'MUNICIPIO', 'ZONA', 'SEXO', 'ARMA MEDIO', ' MODALIDAD PRESUNTA', 'SPOA_CARACTERIZACION', 'CANTIDAD']
————————————————————————————————————————



### ***Detección de columnas inválidas***

In [126]:
# Identificación de columnas con nombres nulos, vacíos o genéricos (Unnamed)

columnas_invalidas = [
    col for col in df.columns
    if (
        col is None
        or pd.isna(col)
        or str(col).strip() == ""
        or "Unnamed" in str(col)
    )
]

print("\n" + "🔍" + "—"*35)
print("REVISIÓN DE NOMBRES DE COLUMNAS")
print("—"*36)

if columnas_invalidas:
    print(f"⚠️  Atención: Se detectaron {len(columnas_invalidas)} columnas inválidas.")
    print(f"Lista: {columnas_invalidas}")
else:
    print("✅ ¡Éxito! Todas las columnas tienen nombres válidos.")
    print("No se encontraron columnas vacías o tipo 'Unnamed'.")

print("—"*36 + "\n")


🔍———————————————————————————————————
REVISIÓN DE NOMBRES DE COLUMNAS
————————————————————————————————————
✅ ¡Éxito! Todas las columnas tienen nombres válidos.
No se encontraron columnas vacías o tipo 'Unnamed'.
————————————————————————————————————



### ***Revisión detallada de integridad***

In [127]:
print("\n" + "🔍" + "—"*40)
print("ANÁLISIS DE COMPLETITUD POR COLUMNA")
print("—"*41)

# 1. Calculamos el conteo y los faltantes
conteo_total = len(df)
datos_presentes = df.count()
faltantes = conteo_total - datos_presentes

# 2. Filtramos solo las que tienen faltantes > 0
columnas_con_huecos = faltantes[faltantes > 0]

if not columnas_con_huecos.empty:
    print(f"⚠️  Se detectaron {len(columnas_con_huecos)} columnas con datos faltantes:\n")
    print(f"{'Columna':<25} | {'Presentes':<10} | {'Faltantes':<10}")
    print("-" * 50)

    for col, nulos in columnas_con_huecos.items():
        presentes = datos_presentes[col]
        print(f"{col:<25} | {presentes:<10} | {nulos:<10}")

    print("\n💡 Sugerencia: Evalúar si estos vacíos, ya que pueden afectar el análisis estadístico.")
else:
    print("💎 ¡Éxito total! Todas las columnas están 100% completas.")
    print(f"Total de registros validados: {conteo_total}")

print("—"*41 + "\n")


🔍————————————————————————————————————————
ANÁLISIS DE COMPLETITUD POR COLUMNA
—————————————————————————————————————————
💎 ¡Éxito total! Todas las columnas están 100% completas.
Total de registros validados: 89548
—————————————————————————————————————————



### ***Identificación de categorías***

In [128]:
df["DEPARTAMENTO"].value_counts()
len(df["DEPARTAMENTO"])

print("Conteo por categoría:")
print(df["DEPARTAMENTO"].value_counts())

print("\nTotal de registros:")
print(len(df["DEPARTAMENTO"]))

Conteo por categoría:
DEPARTAMENTO
VALLE DEL CAUCA       16070
ANTIOQUIA             14432
BOGOTA D.C.            7461
CAUCA                  5299
ATLANTICO              4217
NARIÑO                 3825
NORTE DE SANTANDER     3679
BOLIVAR                3299
CUNDINAMARCA           2801
CORDOBA                2379
MAGDALENA              2347
TOLIMA                 2189
SANTANDER              2179
META                   1914
HUILA                  1894
CESAR                  1851
CHOCO                  1707
RISARALDA              1615
LA GUAJIRA             1353
SUCRE                  1314
QUINDIO                1301
PUTUMAYO               1248
ARAUCA                 1149
CALDAS                 1147
CAQUETA                1067
BOYACA                  607
CASANARE                497
GUAVIARE                234
SAN ANDRES ISLAS        210
VICHADA                 121
AMAZONAS                103
GUAINIA                  20
VAUPES                   19
Name: count, dtype: int64

Total de regis

### ***Tipos de datos***

In [129]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 89548 entries, 0 to 89547
Data columns (total 11 columns):
 #   Column                Non-Null Count  Dtype 
---  ------                --------------  ----- 
 0   FECHA HECHO           89548 non-null  object
 1   COD_DEPTO             89548 non-null  int64 
 2   DEPARTAMENTO          89548 non-null  object
 3   COD_MUNI              89548 non-null  int64 
 4   MUNICIPIO             89548 non-null  object
 5   ZONA                  89548 non-null  object
 6   SEXO                  89548 non-null  object
 7   ARMA MEDIO            89548 non-null  object
 8    MODALIDAD PRESUNTA   89548 non-null  object
 9   SPOA_CARACTERIZACION  89548 non-null  object
 10  CANTIDAD              89548 non-null  int64 
dtypes: int64(3), object(8)
memory usage: 7.5+ MB


### ***Dimensiones***

In [130]:
df.shape

(89548, 11)

### ***Nulos***

In [131]:
# Ver cantidad y porcentaje de valores nulos por columna
nulos = pd.DataFrame({
    'Valores Nulos': df.isnull().sum(),
    'Porcentaje (%)': round((df.isnull().sum() / len(df)) * 100, 2)
})

# Mostrar solo columnas con nulos
nulos = nulos[nulos['Valores Nulos'] > 0]

# Ordenar de mayor a menor
nulos = nulos.sort_values(by='Valores Nulos', ascending=False)

print(nulos)

Empty DataFrame
Columns: [Valores Nulos, Porcentaje (%)]
Index: []


### **Transformación**

La etapa de transformación representó el núcleo del proceso ETL, permitiendo convertir datos crudos en información estructurada y analíticamente útil.

***Convertir Fecha***

In [132]:
df['FECHA HECHO'] = pd.to_datetime(df['FECHA HECHO'])

***Crear Columnas temporales***

In [133]:
df['AÑO'] = df['FECHA HECHO'].dt.year
df['MES'] = df['FECHA HECHO'].dt.month
df['DIA'] = df['FECHA HECHO'].dt.day

In [134]:
df.columns

Index(['FECHA HECHO', 'COD_DEPTO', 'DEPARTAMENTO', 'COD_MUNI', 'MUNICIPIO',
       'ZONA', 'SEXO', 'ARMA MEDIO', ' MODALIDAD PRESUNTA',
       'SPOA_CARACTERIZACION', 'CANTIDAD', 'AÑO', 'MES', 'DIA'],
      dtype='object')

In [135]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 89548 entries, 0 to 89547
Data columns (total 14 columns):
 #   Column                Non-Null Count  Dtype         
---  ------                --------------  -----         
 0   FECHA HECHO           89548 non-null  datetime64[ns]
 1   COD_DEPTO             89548 non-null  int64         
 2   DEPARTAMENTO          89548 non-null  object        
 3   COD_MUNI              89548 non-null  int64         
 4   MUNICIPIO             89548 non-null  object        
 5   ZONA                  89548 non-null  object        
 6   SEXO                  89548 non-null  object        
 7   ARMA MEDIO            89548 non-null  object        
 8    MODALIDAD PRESUNTA   89548 non-null  object        
 9   SPOA_CARACTERIZACION  89548 non-null  object        
 10  CANTIDAD              89548 non-null  int64         
 11  AÑO                   89548 non-null  int32         
 12  MES                   89548 non-null  int32         
 13  DIA             

***Integración de Coordenadas Geográficas por Departamento***

In [136]:
# Unir las coordenadas de df_Departamentos al dataframe principal df

df = df.merge(
    df_Departamentos[['Código Departamento', 'Latitud', 'longitud']],
    left_on='COD_DEPTO',
    right_on='Código Departamento',
    how='left'
)

# Eliminar columna duplicada después del merge
df = df.drop(columns=['Código Departamento'])

# Verificar resultado
print(df.head())

  FECHA HECHO  COD_DEPTO DEPARTAMENTO  COD_MUNI   MUNICIPIO    ZONA  \
0  2017-01-01         91     AMAZONAS     91001     LETICIA   RURAL   
1  2017-01-01          5    ANTIOQUIA      5091     BETANIA  URBANA   
2  2017-01-01          5    ANTIOQUIA      5212  COPACABANA  URBANA   
3  2017-01-01          5    ANTIOQUIA      5240     EBEJICO   RURAL   
4  2017-01-01          5    ANTIOQUIA      5250    EL BAGRE  URBANA   

        SEXO                   ARMA MEDIO                 MODALIDAD PRESUNTA  \
0  MASCULINO  ARMA BLANCA / CORTOPUNZANTE                              RIÑAS   
1  MASCULINO                ARMA DE FUEGO                              RIÑAS   
2  MASCULINO  ARMA BLANCA / CORTOPUNZANTE  RIÑA ENTRE COMPAÑEROS PERMANENTES   
3  MASCULINO                ARMA DE FUEGO                          SICARIATO   
4  MASCULINO  ARMA BLANCA / CORTOPUNZANTE                              RIÑAS   

               SPOA_CARACTERIZACION  CANTIDAD   AÑO  MES  DIA      Latitud  \
0  HOMICIDIO I

**Creación de Variables Categóricas Binarias**

***Variable de zona urbana***

In [137]:
df['ZONA_URBANA'] = df['ZONA'].apply(
    lambda x: 'Si' if x == 'CABECERA MUNICIPAL' else 'No'
)

***Variable de arma de fuego***

In [138]:
df['ARMA_FUEGO'] = df['ARMA MEDIO'].apply(
    lambda x: 'Si' if x == 'ARMA DE FUEGO' else 'No'
)

## **3. Carga y Almacenamiento de Datos (Load)**

Una vez finalizado el proceso ETL, la información procesada fue almacenada en Google Sheets para facilitar su integración con herramientas de visualización.

Se realizó la autenticación de Google Drive y Google Sheets mediante Python:Posteriormente, se instalaron las librerías necesarias:Finalmente, se realizó la conexión y carga de datos.

In [112]:
# Instalar librerías
!pip install --upgrade gspread gspread_dataframe

# Importar librerías
import gspread
from google.colab import auth
from google.auth import default
from gspread_dataframe import set_with_dataframe

# Autenticación manual
auth.authenticate_user()

# Obtener credenciales
creds, _ = default()

# Autorizar acceso
gc = gspread.authorize(creds)

# Crear Google Sheet
sh = gc.create('Dashboard_Homicidios_Colombia')

# Seleccionar hoja
worksheet = sh.get_worksheet(0)

# Exportar DataFrame
set_with_dataframe(worksheet, df)

print("✅ Datos exportados correctamente")

✅ Datos exportados correctamente


## **Construcción de KPIs Estratégicos**

### ***KPI: Total de Homicidios***

In [139]:
total_homicidios = len(df)
print(total_homicidios)

89548


### ***KPI: Homicidios por Departamento***

In [141]:
homicidios_departamento = df.groupby('DEPARTAMENTO')['CANTIDAD'].sum().reset_index()

print(homicidios_departamento)

          DEPARTAMENTO  CANTIDAD
0             AMAZONAS       104
1            ANTIOQUIA     14480
2               ARAUCA      1157
3            ATLANTICO      4231
4          BOGOTA D.C.      7501
5              BOLIVAR      3313
6               BOYACA       607
7               CALDAS      1149
8              CAQUETA      1071
9             CASANARE       497
10               CAUCA      5328
11               CESAR      1858
12               CHOCO      1714
13             CORDOBA      2384
14        CUNDINAMARCA      2809
15             GUAINIA        20
16            GUAVIARE       234
17               HUILA      1895
18          LA GUAJIRA      1356
19           MAGDALENA      2352
20                META      1921
21              NARIÑO      3847
22  NORTE DE SANTANDER      3703
23            PUTUMAYO      1258
24             QUINDIO      1302
25           RISARALDA      1617
26    SAN ANDRES ISLAS       212
27           SANTANDER      2182
28               SUCRE      1316
29        

### ***KPI: Distribución por Sexo***

In [142]:
df['SEXO'].value_counts()

,count
SEXO,
MASCULINO,82331
FEMENINO,7217


### ***KPI: Homicidios con Arma de Fuego***

In [145]:
arma_fuego = df[
    df['ARMA MEDIO'] == 'ARMA DE FUEGO'
].shape[0]
print(arma_fuego)

66669


### ***KPI: Evolución Temporal***

In [146]:
homicidios_anio = df.groupby(
    'AÑO'
)['CANTIDAD'].sum().reset_index()
print(homicidios_anio)

    AÑO  CANTIDAD
0  2017     11957
1  2018     12573
2  2019     12547
3  2020     12010
4  2021     13685
5  2022     13536
6  2023     13555
